In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so
from jax import vmap
from src.experiment import (
    ElectronReactionSamplingExperiment,
    HeterogeneousReactionSamplingExperiment,
)
from src.fdm import (
    AdsorptionReactionNewtonDFSolver,
    ElectronReactionFDSolver,
    HeterogeneousReactionFDSolver,
)
from src.params import (
    AdsorptionReactionParams,
    ElectronReactionParams,
    HeterogenousReactionParams,
)
from src.voltammetry import CyclicDC

sns.set_context("paper", font_scale=1.5)

# Figure 1: Example Voltammagram


In [ ]:
voltammetry = CyclicDC()

fdm_solver = ElectronReactionFDSolver(voltammetry)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

# Irreversible

irre_params = ElectronReactionParams(
    alpha=jnp.array(0.7),
    K0=jnp.array(1.0),
    Ef=jnp.array(0.0),
    dB=jnp.array(1.0),
)

irre_current = fdm_solver.solve(irre_params)

ax1.plot(fdm_solver.applied_potentials, irre_current)
ax1.axhline(
    y=-0.496 * jnp.sqrt(irre_params.alpha) * jnp.sqrt(voltammetry.sigma),
    linestyle="--",
    c="red",
)
ax1.axvline(
    x=(jnp.log(irre_params.K0 / jnp.sqrt(irre_params.alpha * voltammetry.sigma)) - 0.78)
    / irre_params.alpha,
    linestyle="--",
    c="red",
)

# Reversible
rev_params = ElectronReactionParams(
    alpha=jnp.array(0.7),
    K0=jnp.array(100000.0),
    Ef=jnp.array(0.0),
    dB=jnp.array(1.0),
)

rev_current = fdm_solver.solve(rev_params)

ax2.plot(fdm_solver.applied_potentials, rev_current)
ax2.axhline(y=-0.446 * jnp.sqrt(voltammetry.sigma), linestyle="--", c="red")

ax1.set_xlabel(r"$\theta$")
ax1.set_ylabel(r"$J$")
ax2.set_xlabel(r"$\theta$")

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


# Figure 4: Electron Reaction Biplot


In [ ]:
experiment = ElectronReactionSamplingExperiment()

hmc = np.load("./data/E_HMC_CyclicDC.npz")
rw = np.load("./data/E_MetropolisHastings_CyclicDC.npz")

df_hmc = pd.DataFrame({k: hmc[k].flatten() for k in hmc.files})
df_rw = pd.DataFrame({k: rw[k].flatten() for k in rw.files})

df_hmc["algorithm"] = "HMC"
df_rw["algorithm"] = "RW"

df_hmc["algorithm"] = df_hmc["algorithm"].astype("category")
df_rw["algorithm"] = df_rw["algorithm"].astype("category")

sampling_df = pd.concat([df_hmc, df_rw], ignore_index=True)

In [ ]:
g = sns.PairGrid(
    sampling_df.drop(columns=["logdensity"]),
    corner=True,
    layout_pad=True,
    hue="algorithm",
)

g.map_diag(sns.kdeplot)
g.map_lower(sns.kdeplot)

plt.tight_layout()
plt.show()


# Figure 5: Heterogeneous Parameter Effects

In [ ]:
voltammetry = CyclicDC()
fdm_solver = HeterogeneousReactionFDSolver(voltammetry, omega=1.0)

params = HeterogenousReactionParams(
    alpha1=jnp.array(0.7),
    K1_0=jnp.array(1.0),
    E1_f=jnp.array(0.0),
    alpha2=jnp.array(0.0),
    K2_0=jnp.array(0.0),
    E2_f=jnp.array(0.0),
    dB=jnp.array(1.0),
    dC=jnp.array(1.0),
    dD=jnp.array(1.0),
    K_het=jnp.array(0.0),
)

current = fdm_solver.solve(params)

plt.plot(fdm_solver.applied_potentials, current)
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.show()


# Figure 6: Heterogeneous Biplot


In [ ]:
rw = np.load("./data/Heterogenous_MetropolisHastings_CyclicDC.npz")

fig, axs = plt.subplots(2, 5, figsize=(12, 5))
options = {"bins": 50}

axs[0, 0].hist(rw["alpha1"].flatten(), **options)
axs[1, 0].hist(rw["alpha2"].flatten(), **options)
plt.tight_layout()
plt.show()

# Figure 10: Adsorption Example Voltammagrams

In [ ]:
voltammetry = CyclicDC(sigma=10, theta_i=20, theta_v=-20)

fig, axs = plt.subplots(2, 2, figsize=(10, 6))

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.5),
    K0_sol=jnp.array(0.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.5),
    K0_ads=jnp.array(1e6),
    K_A_ads=jnp.array(1e3),
    K_A_des=jnp.array(1e-3),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1e-3),
    dB=jnp.array(0.8),
)

fdm_solver = AdsorptionReactionNewtonDFSolver(voltammetry, h0=1e-6, dtheta=1e-1)
current = fdm_solver.solve(params)

axs[0, 0].plot(fdm_solver.applied_potentials, current)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-3),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(4.5),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
    dB=jnp.array(1.0),
)

current = fdm_solver.solve(params)

axs[0, 1].plot(fdm_solver.applied_potentials, current)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.9),
    K0_sol=jnp.array(1000.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.9),
    K0_ads=jnp.array(0.0),
    K_A_ads=jnp.array(1.0),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
    dB=jnp.array(1.0),
)

current = fdm_solver.solve(params)

axs[1, 0].plot(fdm_solver.applied_potentials, current)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-2),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(1.0),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(5.0),
    K_B_des=jnp.array(1e-1),
    dB=jnp.array(1.0),
)

current = fdm_solver.solve(params)

axs[1, 1].plot(fdm_solver.applied_potentials, current)


plt.show()
